# Flow Matching Inference

In this notebook, we will show the process of generating predictions using a trained flow matching model.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from abc import ABC, abstractmethod

import torch
import xarray as xr
import numpy as np

from src.config import load_config

In [ ]:
CONFIG_PATH = str(Path.cwd().parent) + "/src/config/config.yaml"
DATA_PATH = "/home/users/industry/metoffice/04052003/scratch/singapore-demo-2026/sg128_8km_to_2km"

PREDICTORS_PATH = DATA_PATH + "/predictors"
TARGETS_PATH = DATA_PATH + "/targets"
STATIC_PATH = DATA_PATH + "/static/static_128_sqrt_normalised.nc"

Update the paths below, to point towards the pre-trained model and the transforms which were fitted during training.

In [ ]:
MODEL_PATH = ...
TRANSFORMS_PATH = ...

# # Uncomment if you would like to use a model that has been trained for 500 epochs!
# MODEL_PATH = "/home/users/industry/metoffice/04052003/scratch/singapore-demo-2026/model.pth"
# TRANSFORMS_PATH = "/home/users/industry/metoffice/04052003/scratch/singapore-demo-2026/transforms.json"

Please specify where would you like to any outputs created in this notebook to be stored, by completing `OUTPUT_PATH` below.

In [ ]:
OUTPUT_PATH = ...

Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)
print(f"Created directory to store inference outputs: \n{OUTPUT_PATH}")

In [ ]:
NUM_WORKERS = 0
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

## Test data

In this notebook, we will test the flow matching model on data that was never seen during training. We reserved the decade "2050-01-01" to "2059-12-31" for model testing.  However, for this demo, we will just demonstrate inference over a one year period. You can specify which days you would like to generate predictions for in the config file.

In [ ]:
config = load_config(CONFIG_PATH)
split = "test"

inference_dates = getattr(config, split).date_ranges

print(inference_dates)

## Load transforms

During training, the predictors and targets were transformed (according to the transforms specified in the config file). The names of the transforms and any associated statistics over the training dataset were saved, so that the same transforms can be applied to the inference dataset.

In [ ]:
from src.transforms import Transform, load_transforms

transforms: dict[str, Transform] = load_transforms(TRANSFORMS_PATH)

print(f"Loaded transforms from {TRANSFORMS_PATH}")

## Initialise datasets and dataloaders

The predictors (including any static predictors) and the targets are selected for the inference dates, and the transforms are applied. We prepare the dataset for the model, splitting the data into batches along the time dimension (batch size specified in the config)

In [ ]:
from src.dataset import V3Dataset

inference_dataset = V3Dataset(
    predictors_path=PREDICTORS_PATH,
    targets_path=TARGETS_PATH,
    config=config,
    date_ranges=inference_dates,
    transforms=transforms,
    static_path=STATIC_PATH,
)
batch_size = config.training.batch_size
inference_dataloader = torch.utils.data.DataLoader(
    inference_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

## Load the model

We load the pre-trained flow model from `MODEL_PATH`.

In [ ]:
from src.network import ADM

static_variables = config.static or []
if static_variables:
    print(f"Using static variables: {static_variables}")
else:
    print("Not using static variables")
if config.pos_emb:
    print("Using sinusoidal positional embeddings (4 channels)")
num_conditioning_variables = (
    len(config.predictors) + len(static_variables) + (4 if config.pos_emb else 0)
)

model = ADM(
    num_conditioning_variables=num_conditioning_variables,
    num_target_variables=len(config.targets),
    **config.model.model_dump(),
).to(DEVICE)

In [ ]:
# strip _orig_mod. prefix (artifact of the model trained on azureML)
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

print(f"Loaded model from {MODEL_PATH}")

## Source/initial distribution

We use a standard Gaussian distribution as our source distribution (as in training).

In [ ]:
class GaussianDistribution:
    """Gaussian distribution sampler."""

    def __init__(self, mean: float, std: float, device: torch.device):
        """Initialise Gaussian distribution sampler.

        Args:
            mean: Mean of the Gaussian distribution.
            std: Standard deviation of the Gaussian distribution.
            device: Device to place samples on.
        """
        self.mean = mean
        self.std = std
        self.device = device

    def sample(self, shape: tuple) -> torch.Tensor:
        return torch.normal(self.mean, self.std, size=shape, device=self.device)

In [ ]:
source_sampler = GaussianDistribution(
    mean=config.source.mean, std=config.source.std, device=DEVICE
)

## Numerical integration of the flow ODE

We generate predictions using the trained flow matching model by numerically integrating the ODE
\begin{equation}
    \frac{\mathrm{d}x_t}{\mathrm{d}t} = u_{\theta}(x_t, t; c)\, , \tag{1}
\end{equation}
from $t=0$ to $t=1$, starting from the initial condition
\begin{equation}
    x_0 \sim \mathcal{N}(0, I_d)\, . \tag{2}
\end{equation}
In equation (1), $u_{\theta}$ is the neural network vector field, which is a function of the current state $x_t$, simulation time $t$, and conditioning variables $c$ (large scale atmospheric variables + static fields).

If we change the initial condition (by resampling from the initial distribution (2)), the flow will follow a different trajectory through phase space, which generates a new sample from the data distribution.

The following `Solver` class defines the interface for simulating the learned velocity field. The solver takes as inputs the number of steps for the numerical integration (`n_steps`) as well as a time-stepping parameter `alpha`. When `alpha`$=1$ the time steps are uniformly spaced. When `alpha`$>1$, the steps are smaller near $t=1$, allowing for finer integration near the target distribution.

Subclasses of `Solver` allow the implementaion of different numerical intergation schemes by the specification of the `step()` method.

In [ ]:
class Solver(ABC):
    def __init__(
        self,
        n_steps: int,
        alpha: float,
        model: torch.nn.Module,
        device: torch.device,
    ):
        """Initialise.

        Args:
            n_steps: Number of steps.
            alpha: Exponent for the time-stepping scheme. If alpha=1, then the time steps
                are uniformly spaced between 0 and 1 (constant step size). If alpha>1, then the
                step size is smaller near t=1 (the data/target side) and larger near t=0 (the
                noise/source side).
            model: Neural network model representing the velocity/vector field.
            device: Device
        """
        self.n_steps = n_steps
        self.alpha = alpha
        self.model = model
        self.device = device
        self.time_steps = (
            1 - (1 - torch.tensor(range(self.n_steps + 1)) / self.n_steps) ** self.alpha
        )  # shape (n_steps + 1, )
        self.step_sizes = self.time_steps[1:] - self.time_steps[:-1]  # shape (n_steps, )

    def simulate(
        self, x_0: torch.Tensor, predictors: torch.Tensor, static: torch.Tensor | None
    ) -> torch.Tensor:
        """Simulate the ODE from t=0 to t=1.

        Args:
            x_0: Tensor of shape (batch_size, num_target_variables, height, width)
            predictors: Predictor variables,
                shape (batch_size, num_predictor_variables, height, width)
            static: Static fields (e.g., orography), shape (batch_size, num_static_variables, height, width).
                If static fields are not used, this should be None.

        Returns:
            x_1: Tensor of shape (batch_size, num_target_variables, height, width)
        """
        x_current = x_0.to(self.device)

        for idx, step_size in enumerate(self.step_sizes):
            t_current = self.time_steps[idx].item()
            t_current = torch.full(
                (x_current.shape[0], 1), t_current, device=self.device
            )  # shape (batch_size, 1)
            x_current = self.step(x_current, t_current, predictors, static, step_size.item())

        return x_current

    def _get_velocity(
        self,
        x_current: torch.Tensor,
        t_current: torch.Tensor,
        predictors: torch.Tensor,
        static: torch.Tensor | None,
    ) -> torch.Tensor:
        """Returns the velocity/vector field predicted by the model at (x_current, t_current)."""
        conditioning = predictors if static is None else torch.cat([predictors, static], dim=1)
        return self.model(x_current, t_current, conditioning)

    @abstractmethod
    def step(
        self,
        x_current: torch.Tensor,
        t_current: torch.Tensor,
        predictors: torch.Tensor,
        static: torch.Tensor | None,
        step_size: float,
    ) -> torch.Tensor:
        """Take a single step of the ODE solver.

        Args:
            x_current: Current state, shape (batch_size, num_target_variables, height, width)
            t_current: Current time between 0 and 1, shape (batch_size, 1)
            predictors: Predictor variables,
                shape (batch_size, num_predictor_variables, height, width)
            static: Static fields (e.g., orography), shape (batch_size, num_static_variables, height, width).
                If static fields are not used, this should be None.
            step_size: Step size for this step.

        Returns:
            x_next: Next state, shape (batch_size, num_target_variables, height, width)
        """
        pass

The simplest numerical integration scheme is the forward Euler method:
\begin{equation}
    x_{t+\Delta t} = x_t + \Delta t \cdot u_{\theta}(x_t, t;c)\, , \tag{3}
\end{equation}
which is implemented in the `EulerSolver` class below. 

In [ ]:
class EulerSolver(Solver):
    def step(
        self,
        x_current: torch.Tensor,
        t_current: torch.Tensor,
        predictors: torch.Tensor,
        static: torch.Tensor | None,
        step_size: float,
    ) -> torch.Tensor:
        x_next = x_current + step_size * self._get_velocity(
            x_current, t_current, predictors, static
        )
        return x_next

### EXERCISE: Add Heun's method as a solver

A more complex method is Heun's method, which is a predictor-corrector method. First, a predictor step is taken via the Euler method to make an initial guess for the state at the next time step:
\begin{equation}
    \tilde{x}_{t+\Delta t} = x_t + \Delta t \cdot u_{\theta}(x_t, t; c)\, . \tag{4}
\end{equation}
Next, a corrector step is taken, which uses the average of the velocity at the current time step and the predicted velocty at the next time step:
\begin{equation}
    x_{t+\Delta t} = x_t + \Delta t \cdot \left[\frac{u_{\theta}(x_t, t; c) + u_{\theta}(\tilde{x}_{t+\Delta t}, t+\Delta t; c)}{2}\right]\, . \tag{5}
\end{equation}

It is left as an exercise to complete the `HeunSolver` class below.

In [ ]:
class HeunSolver(Solver):
    def step(
        self,
        x_current: torch.Tensor,
        t_current: torch.Tensor,
        predictors: torch.Tensor,
        static: torch.Tensor | None,
        step_size: float,
    ) -> torch.Tensor:

        # EXERCISE: COMPLETE HEUN'S METHOD

        return x_next

In [ ]:
# REMOVE THIS BEFORE THE DEMO
#
# class HeunSolver(Solver):
#     def step(
#         self,
#         x_current: torch.Tensor,
#         t_current: torch.Tensor,
#         predictors: torch.Tensor,
#         static: torch.Tensor | None,
#         step_size: float,
#     ) -> torch.Tensor:
#
#         # Predictor step
#         x_next_estimate = x_current + step_size * self._get_velocity(
#             x_current, t_current, predictors, static
#         )

#         # Corrector step
#         t_next = t_current + step_size
#         x_next = x_current + step_size * (
#             self._get_velocity(x_current, t_current, predictors, static) +
#             self._get_velocity(x_next_estimate, t_next, predictors, static)
#         ) / 2

#         return x_next

Select the solver type (`"euler"` or `"heun"`), and choose how to discretise $t \in [0, 1]$ for the numerical integration, by defining `n_steps` and `alpha` below.

In [ ]:
solver_type = "euler"
n_steps = 50
alpha = 2

if solver_type == "euler":
    solver = EulerSolver(n_steps=n_steps, alpha=alpha, model=model, device=DEVICE)
elif solver_type == "heun":
    solver = HeunSolver(n_steps=n_steps, alpha=alpha, model=model, device=DEVICE)
else:
    raise ValueError(f"Unsupported solver type: {solver_type}")

## Visualise the generation of one sample

We are now ready to run inference with our flow model. Before running it over the entire `inference_dataset`, the code block below will generate a single sample for a single day, saving the progress of the state $x_t$, as well as the velocity field $u_{\theta}$, for every step of the numerical integration.

Usually (and as we will do later on) we would just use the `simulate()` method within the `Solver` to generate the prediction for $t=1$, but we save each intermediate step here for illustration.

In [ ]:
# Run inference for a single day and sample, saving intermediate states

# Pick a particular time index from the inference dataset
time_idx = 3

predictors, _, _, static = inference_dataset[time_idx]
predictors = predictors.unsqueeze(0).to(DEVICE) # shape (1, num_predictors, lat, lon)
static = static.unsqueeze(0).to(DEVICE) if static.shape[1] > 0 else None

x_0 = source_sampler.sample(
    (1, len(config.targets), predictors.shape[2],predictors.shape[3])
)  # shape (1, num_target_variables, lat, lon)

x_current = x_0.to(DEVICE)
states = [x_current.cpu()]
velocities = []
times = []

model.eval()
with torch.no_grad():
    for idx, step_size in enumerate(solver.step_sizes):
        t_value = solver.time_steps[idx].item()
        t_current = torch.full((1, 1), t_value, device=DEVICE)

        v_current = solver._get_velocity(x_current, t_current, predictors, static)
        x_next = solver.step(x_current, t_current, predictors, static, step_size.item())

        times.append(t_value)
        velocities.append(v_current.cpu())
        states.append(x_next.cpu())

        x_current = x_next

# Final time t=1
times.append(solver.time_steps[-1].item())

times = np.array(times) # (n_steps + 1,)
states = torch.cat(states, dim=0).numpy() # (n_steps +1, num_targets, lat, lon)
velocities = torch.cat(velocities, dim=0).numpy() # (n_steps, num_targets, lat, lon)

# physical (un-normalised) states
target_var_names = list(config.targets.keys())
physical_states = states.copy()
for idx, target in enumerate(target_var_names):
    physical_states[:, idx] = transforms["targets"][target].inverse_transform(
        physical_states[:, idx]
    )

print(f"Generated one prediction for {str(np.datetime_as_string(np.asarray(inference_dataset.time)[time_idx], unit="D"))}")

Below, we display an animation of the data saved during that single inference run. The animation has four panels: the state $x_t$ which is evolved by the model (in normalised units), the predicted velocity field $u_{\theta}$, the simulation time $t$ as a function of step number, and the state $x_t$ after reversing the transforms (physical precipitation).

In [ ]:
from src.plotting import animate_sampling

anim = animate_sampling(
    states=states,
    velocities=velocities,
    times=times,
    physical_states=physical_states,
    lat=inference_dataset.target_lat,
    lon=inference_dataset.target_lon,
    variable="pr",
    target_idx=target_var_names.index("pr"),
    units=inference_dataset.target_attrs["pr"].get("units"),
    alpha=alpha,
    date=str(np.datetime_as_string(np.asarray(inference_dataset.time)[time_idx], unit="D")),
    out_path=f"{OUTPUT_PATH}/sampling_animation.gif",
    fps=6,
    frame_stride=1,
    hold_frames=20,
    dpi=80,
)

In [ ]:
from IPython.display import Image

ANIMATION_PATH = OUTPUT_PATH + "/sampling_animation.gif"

# # Uncomment if you would like to see an animation already created with the 500 epoch trained model!
# ANIMATION_PATH = "/home/users/industry/metoffice/04052003/scratch/singapore-demo-2026/sampling_animation.gif"

Image(filename=ANIMATION_PATH)

## Run inference loop

Having visualised the process of generating a sample for a single day, we now run inference over the entire `inference_dataset`.

For each day, we can generate an arbitrary number of samples. Specify the number of samples (`n_samples`) to generate in the cell block below.

In [ ]:
# Generate multiple samples for all the days in `inference_dataset`.

# Specify the number of samples to generate for each day
n_samples = 6

prediction_batches: list[torch.Tensor] = []
target_batches: list[torch.Tensor] = []

total_days = len(inference_dataloader.dataset)
days_processed = 0

model.eval()
with torch.no_grad():
    for idx, batch in enumerate(inference_dataloader):
        batch_size = batch[0].shape[0]
        days_processed += batch_size
        
        predictors, targets, _, static = batch
    
        target_batches.append(targets.cpu())
        
        samples: list[torch.Tensor] = []  # list of samples of shape (batch_size, num_target_variables, lat, lon)
        for _ in range(n_samples):
            # Resample source for each sample
            x_0 = source_sampler.sample(
                (
                    predictors.shape[0],
                    len(config.targets),
                    predictors.shape[2],
                    predictors.shape[3],
                )
            )  # shape (batch_size, num_target_variables, lat, lon)
    
            predictors = predictors.to(DEVICE)
            x_0 = x_0.to(DEVICE)
            static = static.to(DEVICE) if static.shape[1] > 0 else None
            
            x_1 = solver.simulate(
                x_0=x_0, predictors=predictors, static=static
            )  # shape (batch_size, num_target_variables, lat, lon)
            samples.append(x_1.cpu())
                
        samples_tensor = torch.stack(
            samples, dim=0
        )  # shape (n_samples, batch_size, num_target_variables, lat, lon)
        prediction_batches.append(samples_tensor)

        print(f"Generated {n_samples} samples for days {days_processed}/{total_days}")

predictions = torch.cat(
    prediction_batches, dim=1
).numpy()  # shape (n_samples, time, num_target_variables, lat, lon)

targets_tensor = torch.cat(
    target_batches, dim=0
).numpy()  # shape (time, num_target_variables, lat, lon)

print("Inference complete.")

## Reverse transforms

Currently, the predictions are in their normalised state. We need to reverse the transforms so the predictions become physical values.

In [ ]:
# Inverse any transforms

target_var_names: list[str] = list(config.targets.keys())  # list of target variable names

data_vars: dict[str, xr.DataArray] = {}
for idx, target in enumerate(target_var_names):
    # Inverse any transforms applied
    target_transform = transforms["targets"][target]
    predictions[:, :, idx, :, :] = target_transform.inverse_transform(
        predictions[:, :, idx, :, :]
    )
    targets_tensor[:, idx, :, :] = target_transform.inverse_transform(
        targets_tensor[:, idx, :, :]
    )

    # Propogate relevant attributes from the target variable
    target_attrs = inference_dataset.target_attrs[target]
    attrs = {
        key: target_attrs[key] for key in ["units", "standard_name"] if key in target_attrs
    }

    data_vars[f"pred_{target}"] = (
        ("sample", "input", "time", "grid_latitude", "grid_longitude"),
        np.expand_dims(
            predictions[:, :, idx, :, :], axis=1
        ),  # expand_dims adds 'dummy' input axis
        attrs,
    )

    data_vars[f"target_{target}"] = (
        ("input", "time", "grid_latitude", "grid_longitude"),
        np.expand_dims(targets_tensor[:, idx, :, :], axis=0),
        attrs,
    )
    print(f"Applied inverse transform to target variable '{target}'")

## Save predictions

The predictions are now saved to a netCDF file.

In [ ]:
# Build xarray Dataset and save to netCDF

input_descriptor = "sg128_8km_to_2km_imperfect"

ds = xr.Dataset(
    data_vars=data_vars,
    coords={
        "sample": np.arange(n_samples),
        "input": [input_descriptor],
        "time": inference_dataset.time,
        "grid_latitude": inference_dataset.target_lat,
        "grid_longitude": inference_dataset.target_lon,
    },
    attrs={
        "solver_type": solver_type,
        "n_steps": n_steps,
        "alpha": alpha,
    },
)

output_file = f"{OUTPUT_PATH}/predictions.nc"
ds.to_netcdf(output_file, format="NETCDF4", engine="netcdf4")
print(f"Predictions saved to {output_file}")

## Display results

We load the `predictions.nc` file and display the results!

In [ ]:
PREDICTIONS_PATH = OUTPUT_PATH + "/predictions.nc"

# # Uncomment if you would like to see the predictions made by the 500 epoch trained model, for 2050-01-01 to 2059-12-31
# PREDICTIONS_PATH = "/home/users/industry/metoffice/04052003/scratch/singapore-demo-2026/predictions-test-set.nc"

In [ ]:
predictions_ds = xr.open_dataset(PREDICTIONS_PATH)
print(predictions_ds)

In [ ]:
from src.plotting import plot_target_vs_predictions

plot_target_vs_predictions(
    predictions_ds,
    time_idx=3,
    variable="pr",
    input_idx=0,
    n_samples=6,
    max_sample_cols=3,
)